In [ ]:
from data import load_dataset_from_str, minmaxscale
from tddpm.data_generation import random_region
from tddpm.crop import crop_signal_with_rotation
from synthetic_loader import find_synthetic_datasets, load_synthetic
from preprocess_data import meters_from_point
from preproc import get_limit
from matplotlib.patches import Rectangle
from config import nice_names

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from cycler import cycler

In [ ]:
def get_limits_meters(dataset_str):
    limits = get_limit(dataset_str)
    origin = [limits[0], limits[2]]  # [lat, lon] of bottom left corner
    limits_meters = meters_from_point(np.array([
        [limits[0], limits[2]],  # bottom left
        [limits[1], limits[3]]   # top right
    ]), origin)
    limits_meters = limits_meters.flatten()
    return limits_meters

In [ ]:
all_datasets = find_synthetic_datasets()

In [ ]:
dataset_strs = ['porto-100', 'cabspotting-100']
models = ['tddpm', 'diffusionts', 'difftraj', 'timegan-forward-seed42', 'timevae', 'coscigan']
seq_lens = [128]
seq_len = seq_lens[0]

In [ ]:
train_datas = []
for dataset_str in dataset_strs:
    train_data_raw = load_dataset_from_str(dataset_str=dataset_str, prefix='data/')
    train_datas.append(train_data_raw)

In [ ]:
def inverse_scale(scaled, data_range):
    d_min, d_max = data_range
    return scaled * (d_max - d_min) + d_min

In [ ]:
inverse_scales = []
regions = []
for train_data_raw in train_datas:

    d_min = np.min(train_data_raw, axis=(0, 1), keepdims=True)
    d_max = np.max(train_data_raw, axis=(0, 1), keepdims=True)

    inverse_scales.append((d_min, d_max))

In [ ]:
model = 'tddpm'
text_size = (5.50107, 9.00177)
figsize = (text_size[0], text_size[1]*0.7)
plot_samples = len(train_datas[0])
# plot_samples = 1000

colors = ['#001219', '#005f73', '#0a9396', '#94d2bd', '#e9d8a6', '#ee9b00', '#ca6702', '#bb3e03', '#ae2012', '#9b2226']
cc = cycler(color=colors)
plt.rc('axes', prop_cycle=cc)
grid = True
save = 'png'

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "text.latex.preamble": r"\usepackage{amsmath}",
    "pgf.texsystem": "pdflatex",
    "pgf.rcfonts": False
})


line_params = {'linewidth': 1}


for i, (dataset_str, data_range) in enumerate(zip(dataset_strs, inverse_scales)):

    select = list(filter(lambda x: x.dataset == dataset_str and x.seq_len == seq_len and x.model_name == model, all_datasets))
    if len(select) == 0:
        continue
    dataset = select[0]
    synthetic_data = load_synthetic(dataset)
    synthetic_data = inverse_scale(synthetic_data, data_range)
    limits = get_limits_meters(dataset_str)  # [min_y, min_x, max_y, max_x]

    train_data = train_datas[i]
    
    f, axs = plt.subplots(nrows=2, ncols=2, figsize=figsize, sharex=True, sharey=True)

    for traj in train_data[:plot_samples]:
        axs[0, 0].plot(traj[:, 1], traj[:, 0], **line_params)

    all_concat = np.concatenate(train_data, axis=0)

    train_data_flattened = train_data.reshape(-1, 2)

    aspect_ratio = (limits[3] - limits[1]) / (limits[2] - limits[0])
    bins_width = 200
    bins = (int(aspect_ratio*bins_width), bins_width)

    hm, edges = np.histogramdd(train_data_flattened, bins=bins, range=([limits[0], limits[2]], [limits[1], limits[3]]))
    hm = np.log(hm + 1e-1)
    axs[0, 1].imshow(hm, origin='lower', extent=[limits[1], limits[3], limits[0], limits[2]])
    
    for traj in synthetic_data[:plot_samples]:
        axs[1, 0].plot(traj[:, 1], traj[:, 0], **line_params)

    synth_data_flattened = synthetic_data.reshape(-1, 2)
    hm, edges = np.histogramdd(synth_data_flattened, bins=bins, range=([limits[0], limits[2]], [limits[1], limits[3]]))
    hm = np.log(hm + 1e-1)
    axs[1, 1].imshow(hm, origin='lower', extent=[limits[1], limits[3], limits[0], limits[2]])

    for ax in axs.flatten():
        ax.set_aspect('equal')
        ax.xaxis.set_ticks_position('none')
        ax.yaxis.set_ticks_position('none')
        ax.xaxis.set_ticklabels([])
        ax.yaxis.set_ticklabels([])

    axs[0, 0].set_title('Original samples')
    axs[1, 0].set_title('Synthetic samples')
    axs[0, 1].set_title('Original log-marginal')
    axs[1, 1].set_title('Synthetic log-marginal')
    
    # plt.tight_layout()
    if save == 'png':
        plt.savefig(f'figures/{dataset_str}_{model}_global_comparison.png', dpi=600, bbox_inches='tight', pad_inches=0.01)
    elif save == 'pdf':
        plt.savefig(f'figures/{dataset_str}_{model}_global_comparison.pdf', dpi=300)
    else:
        plt.show()
    
# Figure description: Comparison between original (top) and synthetic (bottom) trajectories.
# Left panels show individual trajectory samples while right panels show log-density heatmaps
# of all points in the dataset. The synthetic samples closely match both the individual
# trajectory patterns and overall density distribution of the original data.


In [ ]:
text_size = (5.50107, 9.00177)
figsize = (text_size[0], text_size[1]*0.2)
plot_samples = len(train_datas[0])
# plot_samples = 1000

colors = ['#001219', '#005f73', '#0a9396', '#94d2bd', '#e9d8a6', '#ee9b00', '#ca6702', '#bb3e03', '#ae2012', '#9b2226']
cc = cycler(color=colors)
plt.rc('axes', prop_cycle=cc)
grid = True
save = 'png'
fontsize = 8

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "text.latex.preamble": r"\usepackage{amsmath}",
    "pgf.texsystem": "pdflatex",
    "pgf.rcfonts": False
})


line_params = {'linewidth': 0.2}


for i, (dataset_str, data_range) in enumerate(zip(dataset_strs, inverse_scales)):
    train_data = train_datas[i]
    limits = get_limits_meters(dataset_str)
    
    f, axs = plt.subplots(nrows=2, ncols=1+len(models), figsize=figsize, sharex=True, sharey=True)
    axs = axs.T

    for traj in train_data[:plot_samples]:
        axs[0, 0].plot(traj[:, 1], traj[:, 0], **line_params)

    all_concat = np.concatenate(train_data, axis=0)

    train_data_flattened = train_data.reshape(-1, 2)

    aspect_ratio = (limits[3] - limits[1]) / (limits[2] - limits[0])
    bins_width = 200
    bins = (int(aspect_ratio*bins_width), bins_width)

    hm, edges = np.histogramdd(train_data_flattened, bins=bins, range=([limits[0], limits[2]], [limits[1], limits[3]]))
    hm = np.log(hm + 1e-1)
    axs[0, 1].imshow(hm, origin='lower', extent=[limits[1], limits[3], limits[0], limits[2]])

    axs[0, 0].set_title('Original', fontsize=fontsize)
    axs[0, 0].yaxis.set_label_text('Samples', fontsize=fontsize)
    axs[0, 1].yaxis.set_label_text('Log-marginal', fontsize=fontsize)

    for j, model in enumerate(models):
        select = list(filter(lambda x: x.dataset == dataset_str and x.seq_len == seq_len and x.model_name == model, all_datasets))
        if len(select) == 0:
            continue
        dataset = select[0]
        synthetic_data = load_synthetic(dataset)
        synthetic_data = inverse_scale(synthetic_data, data_range)

        for traj in synthetic_data[:plot_samples]:
            axs[1+j, 0].plot(traj[:, 1], traj[:, 0], **line_params)

        synth_data_flattened = synthetic_data.reshape(-1, 2)
        hm, edges = np.histogramdd(synth_data_flattened, bins=bins, range=([limits[0], limits[2]], [limits[1], limits[3]]))
        hm = np.log(hm + 1e-1)
        axs[1+j, 1].imshow(hm, origin='lower', extent=[limits[1], limits[3], limits[0], limits[2]])
        axs[1+j, 0].set_title(nice_names[model], fontsize=fontsize)

    for ax in axs.flatten():
        ax.set_aspect('equal')
        ax.xaxis.set_ticks_position('none')
        ax.yaxis.set_ticks_position('none')
        ax.xaxis.set_ticklabels([])
        ax.yaxis.set_ticklabels([])
    
    # plt.tight_layout()
    if save == 'png':
        plt.savefig(f'figures/{dataset_str}_{model}_small_comparison.png', dpi=600, bbox_inches='tight', pad_inches=0.01)
    elif save == 'pdf':
        plt.savefig(f'figures/{dataset_str}_{model}_small_comparison.pdf', dpi=300)
    else:
        plt.show()
    
# Figure description: Comparison between original (top row) and synthetic (bottom rows) trajectories. Left panels show individual trajectory samples while right panels show log-density heatmaps of all points in the dataset. The synthetic samples are shown for different models, with each row corresponding to a different model's output. The proposed model, TDDM, model (second row) most closely matches both the individual trajectory patterns and overall density distribution of the original data.

In [ ]:
set([dataset.model_name for dataset in all_datasets])

In [ ]:
text_size = (5.50107, 9.00177)
figsize = (text_size[0], text_size[1]*0.7)
plot_samples = len(train_datas[0])
# plot_samples = 1000

colors = ['#001219', '#005f73', '#0a9396', '#94d2bd', '#e9d8a6', '#ee9b00', '#ca6702', '#bb3e03', '#ae2012', '#9b2226']
cc = cycler(color=colors)
plt.rc('axes', prop_cycle=cc)
grid = True
save = 'png'
model = 'tddpm-generalize-from-25-bl-v2'

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "text.latex.preamble": r"\usepackage{amsmath}",
    # Add these lines
    "pgf.texsystem": "pdflatex",
    "pgf.rcfonts": False
})


line_params = {'linewidth': 1}


for i, (dataset_str, data_range) in enumerate(zip(dataset_strs, inverse_scales)):

    select = list(filter(lambda x: x.dataset == dataset_str and x.seq_len == seq_len and x.model_name == model, all_datasets))
    if len(select) == 0:
        continue
    dataset = select[0]
    synthetic_data = load_synthetic(dataset)
    synthetic_data = inverse_scale(synthetic_data, data_range)
    limits = get_limits_meters(dataset_str)  # [min_y, min_x, max_y, max_x]

    train_data = train_datas[i]
    
    f, axs = plt.subplots(nrows=2, ncols=2, figsize=figsize, sharex=True, sharey=True)

    for traj in train_data[:plot_samples]:
        axs[0, 0].plot(traj[:, 1], traj[:, 0], **line_params)

    all_concat = np.concatenate(train_data, axis=0)

    train_data_flattened = train_data.reshape(-1, 2)

    aspect_ratio = (limits[3] - limits[1]) / (limits[2] - limits[0])
    bins_width = 200
    bins = (int(aspect_ratio*bins_width), bins_width)

    hm, edges = np.histogramdd(train_data_flattened, bins=bins, range=([limits[0], limits[2]], [limits[1], limits[3]]))
    hm = np.log(hm + 1e-1)
    axs[0, 1].imshow(hm, origin='lower', extent=[limits[1], limits[3], limits[0], limits[2]])
    
    for traj in synthetic_data[:plot_samples]:
        axs[1, 0].plot(traj[:, 1], traj[:, 0], **line_params)

    synth_data_flattened = synthetic_data.reshape(-1, 2)
    hm, edges = np.histogramdd(synth_data_flattened, bins=bins, range=([limits[0], limits[2]], [limits[1], limits[3]]))
    hm = np.log(hm + 1e-1)
    axs[1, 1].imshow(hm, origin='lower', extent=[limits[1], limits[3], limits[0], limits[2]])

    for ax in axs.flatten():
        ax.set_xlim(limits[1], limits[3])
        ax.set_ylim(limits[0], limits[2])
        ax.set_aspect('equal')
        # continue
        ax.xaxis.set_ticks_position('none')
        ax.yaxis.set_ticks_position('none')
        ax.xaxis.set_ticklabels([])
        ax.yaxis.set_ticklabels([])

    axs[0, 0].set_title('Original samples')
    axs[1, 0].set_title('Synthetic samples')
    axs[0, 1].set_title('Original log-marginal')
    axs[1, 1].set_title('Synthetic log-marginal')

    limits_train = get_limits_meters(dataset_str.replace('-100', '-25'))
    # axs[0, 0].set_xlim(limits_generalization[1], limits_generalization[3])
    # axs[0, 0].set_ylim(limits_generalization[0], limits_generalization[2])
    offset = 75  # Small offset to show all egdes of rectangle
    for ax in axs.flatten(): 
        rect = Rectangle((limits_train[1] + offset, limits_train[0] + offset),
                        limits_train[3]-limits_train[1], 
                        limits_train[2]-limits_train[0],
                        fill=False,
                        linestyle='--',
                        edgecolor='red',
                        linewidth=1)
        ax.add_patch(rect)
    
    # plt.tight_layout()
    if save == 'png':
        plt.savefig(f'figures/{dataset_str}_{model}_generalization_comparison.png', dpi=600, bbox_inches='tight', pad_inches=0.01)
    elif save == 'pdf':
        plt.savefig(f'figures/{dataset_str}_{model}_generalization_comparison.pdf', dpi=300)
    else:
        plt.show()
    
# Figure description: Comparison between original (top) and synthetic (bottom) trajectories.
# Left panels show individual trajectory samples while right panels show log-density heatmaps
# of all points in the dataset. The synthetic samples closely match both the individual
# trajectory patterns and overall density distribution of the original data.
